In [3]:
import glob, os, zipfile, tempfile
import numpy as np
import pandas as pd
import xarray as xr

OUTPUT_DIR = "Code Outputs/Climate Data Extraction Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)

# Configuration
CLIMATE_ROOT = "Copernicus Data"          
END_YEAR     = 2025                       # last year to include 
YEARS        = range(1992, END_YEAR + 1)  # 1992 .. END_YEAR inclusive
INSPECT_ONLY = False                      

# Per-lake bounding boxes (lat_min, lat_max, lon_min, lon_max), approximate lake-surface extents.
LAKE_BOXES = {
    "Lake Victoria":   (-3.10,  0.60, 31.50, 34.90),
    "Lake Tanganyika": (-8.90, -3.30, 29.00, 31.20),
    "Lake Malawi":     (-14.60, -9.40, 33.90, 35.30),
    "Lake Turkana":    ( 2.40,  4.70, 35.80, 36.80),
    "Lake Albert":     ( 1.00,  2.40, 30.30, 31.50),
    "Lake Edward":     (-0.70, -0.05, 29.20, 30.00),
    "Lake Kivu":       (-2.60, -1.50, 28.80, 29.50),
}



def _open_any(path):
    """Open a CDS download whether it is a true netCDF OR a ZIP archive (the
    CDS returns a .nc-named ZIP when several variables are requested together).
    Returns an in-memory dataset (so temp files can be cleaned up)."""
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic[:2] == b"PK":                       # ZIP signature -> extract & merge
        tmp = tempfile.mkdtemp()
        with zipfile.ZipFile(path) as z:
            ncs = [n for n in z.namelist() if n.endswith(".nc")]
            z.extractall(tmp)
        parts = []
        for n in ncs:
            d = xr.open_dataset(os.path.join(tmp, n))
            parts.append(d.load()); d.close()
        return xr.merge(parts, compat="override")
    # true netCDF: try the available engines in order
    for eng in ["netcdf4", "h5netcdf", "scipy"]:
        try:
            d = xr.open_dataset(path, engine=eng)
            out = d.load(); d.close(); return out
        except Exception:
            continue
    raise IOError(f"Could not open {path} with any engine")

def open_std(path):
    """Open a netCDF/ZIP file and rename coords to lat/lon/time regardless of the
    exact ERA5 convention (latitude/valid_time/etc.). Drops the ensemble
    'number' coordinate if present."""
    ds = _open_any(path)
    if "number" in ds.dims:                      
        ds = ds.isel(number=0, drop=True)
    elif "number" in ds.coords:
        ds = ds.reset_coords("number", drop=True)
    ren = {}
    for cand in ["latitude", "lat", "Latitude"]:
        if cand in ds.coords or cand in ds.dims: ren[cand] = "lat"; break
    for cand in ["longitude", "lon", "Longitude"]:
        if cand in ds.coords or cand in ds.dims: ren[cand] = "lon"; break
    for cand in ["valid_time", "time", "forecast_reference_time"]:
        if cand in ds.coords or cand in ds.dims: ren[cand] = "time"; break
    return ds.rename(ren)

def pick_var(ds, candidates):
    """Return the first matching variable short-name present in the dataset."""
    for c in candidates:
        if c in ds.data_vars: return c
    return None

def box_mean(da, box):
    """Spatial mean of a DataArray over a (lat_min,lat_max,lon_min,lon_max) box.
    Uses boolean masking so it is robust to ascending/descending lat order."""
    la0, la1, lo0, lo1 = box
    sub = da.where((da.lat >= la0) & (da.lat <= la1) &
                   (da.lon >= lo0) & (da.lon <= lo1), drop=True)
    return sub.mean(dim=["lat", "lon"], skipna=True).squeeze(drop=True).to_series()


# Inspection, open the first available file of each type and print structure

if INSPECT_ONLY:
    for sub, pat in [("2m Temp Data", "ERA5_2mTemp_*.nc"),
                     ("Precip and Evap Data", "ERA5_Precip_Evap_*.nc")]:
        hits   = glob.glob(os.path.join(CLIMATE_ROOT, sub, "*", pat))
        all_nc = glob.glob(os.path.join(CLIMATE_ROOT, sub, "*", "*.nc"))
        print(f"\n##### {sub}: {len(all_nc)} .nc files total "
              f"({len(hits)} match '{pat}'; the rest are single-variable/other names)")
        if hits:
            with open(sorted(hits)[0], "rb") as f: sig = f.read(4)
            print("  filetype   :", "ZIP (multi-var, will extract)" if sig[:2] == b"PK"
                  else "netCDF/HDF")
            ds = open_std(sorted(hits)[0])
            print("  file       :", os.path.basename(sorted(hits)[0]))
            print("  data_vars  :", list(ds.data_vars))
            print("  coords     :", list(ds.coords))
            for v in ds.data_vars:
                u = ds[v].attrs.get("units", "?")
                print(f"    {v}: units={u}, shape={ds[v].shape}")
            ds.close()
    print("\nIf names look right (t2m / tp / pev, lat/lon/time), set "
          "INSPECT_ONLY = False and re-run.")
    raise SystemExit


# Extract temp (one file per year)
temp_daily = {lk: [] for lk in LAKE_BOXES}
for y in YEARS:
    p = os.path.join(CLIMATE_ROOT, "2m Temp Data", str(y), f"ERA5_2mTemp_{y}.nc")
    if not os.path.exists(p):
        print("  [skip temp]", p); continue
    ds = open_std(p); v = pick_var(ds, ["t2m", "2m_temperature", "t2m_mean"])
    for lk, box in LAKE_BOXES.items():
        s = box_mean(ds[v], box) - 273.15            # Kelvin -> Celsius
        temp_daily[lk].append(s)
    ds.close(); print("  temp", y, "ok")


# extarct potential evaporation and precipitation (two files per year: H1, H2)
prec_daily = {lk: [] for lk in LAKE_BOXES}
pet_daily  = {lk: [] for lk in LAKE_BOXES}
for y in YEARS:
    base = os.path.join(CLIMATE_ROOT, "Precip and Evap Data", str(y))
    # Normal years: combined H1/H2 files (each holds BOTH tp and pev).
    combined = [os.path.join(base, f"ERA5_Precip_Evap_{y}_{h}.nc") for h in ["H1", "H2"]]
    combined = [c for c in combined if os.path.exists(c)]
    if combined:
        # (precip_file, evap_file) - same file supplies both variables
        file_pairs = [(c, c) for c in combined]
    else:
        # Fallback for 1992: tp and pev are in SEPARATE single-variable files
        sp = os.path.join(base, f"ERA5_Precip_{y}.nc")
        se = os.path.join(base, f"ERA5_Evap_{y}.nc")
        if os.path.exists(sp) or os.path.exists(se):
            file_pairs = [(sp if os.path.exists(sp) else None,
                           se if os.path.exists(se) else None)]
        else:
            print("  [skip p/e]", base); continue

    for pf, ef in file_pairs:
        dsp = open_std(pf) if pf else None
        # reuse the same dataset when one combined file supplies both variables
        dse = dsp if (ef == pf and pf is not None) else (open_std(ef) if ef else None)
        for lk, box in LAKE_BOXES.items():
            if dsp is not None:
                vp = pick_var(dsp, ["tp", "total_precipitation"])
                if vp: prec_daily[lk].append(box_mean(dsp[vp], box) * 1000.0)  # m -> mm
            if dse is not None:
                ve = pick_var(dse, ["pev", "potential_evaporation"])
                if ve: pet_daily[lk].append(box_mean(dse[ve], box) * 1000.0)   # m -> mm
        print("  p/e", y, "ok")


# Agrregate to monthly and assemble table
#    temp -> monthly mean ; precip & PET -> monthly SUM (mm/month)
frames = []
for lk in LAKE_BOXES:
    t = pd.concat(temp_daily[lk]).sort_index() if temp_daily[lk] else pd.Series(dtype=float)
    pr = pd.concat(prec_daily[lk]).sort_index() if prec_daily[lk] else pd.Series(dtype=float)
    pe = pd.concat(pet_daily[lk]).sort_index()  if pet_daily[lk]  else pd.Series(dtype=float)
    # ERA5 potential evaporation is negative (upward flux); make it positive loss
    if len(pe) and pe.mean() < 0: pe = -pe
    tm  = t.resample("MS").mean()
    prm = pr.resample("MS").sum()
    pem = pe.resample("MS").sum()
    df_out = pd.DataFrame({"temp_C": tm, "precip_mm": prm, "pet_mm": pem})
    df_out["water_balance_mm"] = df_out["precip_mm"] - df_out["pet_mm"]
    df_out["Reservoir"] = lk; df_out["Date"] = df_out.index
    frames.append(df_out.reset_index(drop=True))

climate = pd.concat(frames, ignore_index=True)[
    ["Date", "Reservoir", "precip_mm", "pet_mm", "temp_C", "water_balance_mm"]]
climate.to_excel(out("Lake_Climate_Monthly.xlsx"), index=False)
print("\nWrote Lake_Climate_Monthly.xlsx", climate.shape)
print(climate.groupby("Reservoir").size())

  temp 1992 ok
  temp 1993 ok
  temp 1994 ok
  temp 1995 ok
  temp 1996 ok
  temp 1997 ok
  temp 1998 ok
  temp 1999 ok
  temp 2000 ok
  temp 2001 ok
  temp 2002 ok
  temp 2003 ok
  temp 2004 ok
  temp 2005 ok
  temp 2006 ok
  temp 2007 ok
  temp 2008 ok
  temp 2009 ok
  temp 2010 ok
  temp 2011 ok
  temp 2012 ok
  temp 2013 ok
  temp 2014 ok
  temp 2015 ok
  temp 2016 ok
  temp 2017 ok
  temp 2018 ok
  temp 2019 ok
  temp 2020 ok
  temp 2021 ok
  temp 2022 ok
  temp 2023 ok
  temp 2024 ok
  temp 2025 ok
  p/e 1992 ok
  p/e 1993 ok
  p/e 1993 ok
  p/e 1994 ok
  p/e 1994 ok
  p/e 1995 ok
  p/e 1995 ok
  p/e 1996 ok
  p/e 1996 ok
  p/e 1997 ok
  p/e 1997 ok
  p/e 1998 ok
  p/e 1998 ok
  p/e 1999 ok
  p/e 1999 ok
  p/e 2000 ok
  p/e 2000 ok
  p/e 2001 ok
  p/e 2001 ok
  p/e 2002 ok
  p/e 2002 ok
  p/e 2003 ok
  p/e 2003 ok
  p/e 2004 ok
  p/e 2004 ok
  p/e 2005 ok
  p/e 2005 ok
  p/e 2006 ok
  p/e 2006 ok
  p/e 2007 ok
  p/e 2007 ok
  p/e 2008 ok
  p/e 2008 ok
  p/e 2009 ok
  p/e 2009 ok
